# Homework 1 — AI or Not? Exploring Engineering Data
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · **Due:** Sunday 11:59 pm (Arizona time) at the end of Week 3, on Brightspace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/homework/hw1-ai-or-not/HW1_AI_or_Not.ipynb)

This homework has four parts. **Parts A and B** check that you can use the course vocabulary precisely: what counts as AI (autonomy and adaptivity), and which kind of machine learning fits a task. **Part C** applies your Lab 02 pandas skills to a new dataset from building energy (ARCE). **Part D** asks what AI could mean in your own studies. Read the full handout (README) on GitHub for the rubric. Total: **100 points**; plan about **4 hours**. The AI-use acknowledgment at the end is required: a missing acknowledgment costs 5 points.

### What you will be able to do
1. Judge how much autonomy and adaptivity a system has, and justify it with the course definitions.
2. Classify engineering tasks as supervised (regression or classification), unsupervised, or reinforcement learning, and name their inputs and outputs.
3. Explore a new engineering dataset with pandas and matplotlib, and explain what the plots show.
4. Reflect on where AI could help in civil or architectural engineering — and where it needs checking.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `building_energy_efficiency.csv` — 768 simulated building shapes (Tsanas & Xifara 2012, UCI, CC BY 4.0). See the course `data/` folder README.

**AI-use label:** **Mixed — see each part.** Parts A, B, and D are **AI-Free**: do not use AI tools to produce your answers (you may use them beforehand to study). Part C is **AI-Assisted**: you may ask an AI tool for coding or debugging help; say so in the acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part A — The autonomy–adaptivity map (24 points · AI-Free)
From the Week 1 lecture (after *Elements of AI*, University of Helsinki):
- **Autonomy** — the ability to perform tasks in complex environments **without constant guidance by a user**.
- **Adaptivity** — the ability to **improve performance by learning from experience**.

The more a system shows *both*, the more it counts as AI. Careful: *reacting* to a sensor is not the same as
*learning* — a system that follows fixed rules does not become better with experience.

| # | System |
|---|---|
| 1 | BIM clash detection: checks a building model for pipes, ducts, and beams that intersect, using fixed geometric rules, when a designer clicks “Run.” |
| 2 | Occupancy-sensor lighting in a classroom: lights turn on when motion is detected and off after 15 minutes with no motion. |
| 3 | Survey drone: flies a pre-programmed grid over a construction site, steers around obstacles with its sensors, and returns to land by itself when the battery is low. |
| 4 | Water-main risk model: trained on 10 years of a city's pipe-break records; ranks every pipe by its chance of breaking next year. An asset manager reviews the list and decides which pipes to replace. |
| 5 | Learning thermostat: learns a household's schedule and preferred temperatures from their past adjustments, then sets the temperature on its own. |
| 6 | AI chat assistant (like ChatGPT) used by a student to answer questions about the building code, one prompt at a time. |
| 7 | Adaptive traffic signals on a corridor: change green times every cycle based on vehicle-detector counts, following rules written by traffic engineers. |
| 8 | Site-monitoring robot: walks a construction site every night, scans it with lidar and cameras, compares progress with the BIM schedule, and is retrained every month on scans that engineers corrected, so it recognizes building elements better over time. |

**A1.** Give each system a score from **0 (none) to 10 (very high)** for autonomy and for adaptivity. Then run the next cell to draw your map.

> 💡 **Hint:** Ask two separate questions for each system: (1) Does it do its job without a person guiding each step, in a complex setting? (2) Does it get better by learning from data or experience?

In [ ]:
# TODO (A1): replace each None with a number from 0 to 10.   (autonomy, adaptivity)
scores = {
    "1 Clash detection":   (None, None),
    "2 Occupancy lights":  (None, None),
    "3 Survey drone":      (None, None),
    "4 Pipe-risk model":   (None, None),
    "5 Learning thermostat": (None, None),
    "6 Chat assistant":    (None, None),
    "7 Adaptive signals":  (None, None),
    "8 Site robot":        (None, None),
}

In [ ]:
def plot_map(scores):
    """Draw the autonomy-adaptivity map from a dict of (autonomy, adaptivity) scores."""
    plt.figure(figsize=(7, 6))
    plt.axhline(5, color="gray", linewidth=1)
    plt.axvline(5, color="gray", linewidth=1)
    placed = []   # points already drawn, so labels of nearby points can be stacked instead of overlapping
    for name, (aut, ada) in scores.items():
        if aut is None or ada is None:
            continue
        n_near = sum(1 for (a2, d2) in placed if abs(a2 - aut) < 0.8 and abs(d2 - ada) < 3)
        placed.append((aut, ada))
        plt.scatter(ada, aut, s=80, color="#AB0520", zorder=3)
        plt.annotate(name, (ada, aut), textcoords="offset points", xytext=(6, 6 - 14 * n_near), fontsize=9)
    for (tx, ty, label) in [(0.3, 9.6, "high autonomy\nlow adaptivity"), (5.3, 9.6, "high autonomy\nhigh adaptivity"),
                            (0.3, 0.3, "low autonomy\nlow adaptivity"), (5.3, 0.3, "low autonomy\nhigh adaptivity")]:
        plt.text(tx, ty, label, fontsize=8, color="gray", va="top" if ty > 5 else "bottom")
    plt.xlim(-0.5, 10.5); plt.ylim(-0.5, 10.5)
    plt.xlabel("Adaptivity (0 = never learns, 10 = keeps learning from experience)")
    plt.ylabel("Autonomy (0 = needs constant guidance, 10 = acts alone)")
    plt.title("My autonomy–adaptivity map")
    plt.show()

plot_map(scores)

**A2.** For each system, state low / medium / high for autonomy and for adaptivity, with a one-sentence reason that uses the definitions. Edit the table below (double-click it).

| # | Autonomy (low / medium / high) and why | Adaptivity (low / medium / high) and why |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |
| 7 | | |
| 8 | | |

## Part B — What kind of learning is it? (18 points · AI-Free)
- **Supervised learning** — learn from examples *with the right answers* (labels). **Regression** predicts a number;
  **classification** predicts a category.
- **Unsupervised learning** — find structure in data *without labels* (e.g., **clustering**, **anomaly detection**).
- **Reinforcement learning** — an agent learns *by trial and error* from rewards or penalties for its actions.

| # | Task |
|---|---|
| 1 | Predict the 28-day compressive strength (MPa) of a new concrete mix from its ingredient amounts, using 1,000 past laboratory tests. |
| 2 | Label photos of bridge decks as “spalling” or “no spalling,” using 5,000 photos that inspectors already labeled. |
| 3 | Group 2,000 campus buildings by the shape of their daily electricity-use curve, with no labels given, to find buildings that behave alike. |
| 4 | A building controller learns, by trial and error over many simulated summer days, when to pre-cool the building to lower energy cost while keeping occupants comfortable. |
| 5 | Flag unusual vibration readings from sensors on a pedestrian bridge that might signal damage — no damage has ever been recorded on this bridge. |
| 6 | Estimate the average daily traffic (vehicles/day) on road segments without counters, from population, land use, and road type, using 300 road segments that do have counters. |

For each task, give the type of learning, the sub-type (regression / classification / clustering / anomaly
detection / —), and the input(s) and output — or, for unsupervised and reinforcement learning, say why it is not
supervised. Edit the table below.

| # | Type | Sub-type | Input(s) → output, or why |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |
| 6 | | | |

## Part C — Exploring building energy data (40 points · AI-Assisted)
The dataset holds **768 simulated buildings**. They have the same materials and differ only in shape, glazing
(windows), and orientation. For each one, an energy simulation computed the yearly **heating load** and **cooling
load** in kWh per m² (energy needed to keep the building comfortable).

| Column | Meaning | Units |
|---|---|---|
| `relative_compactness` | Volume-to-surface ratio compared with the most compact shape (higher = more compact) | – |
| `surface_area_m2`, `wall_area_m2`, `roof_area_m2` | Envelope areas | m² |
| `overall_height_m` | Building height (3.5 or 7) | m |
| `orientation` | 2 = North, 3 = East, 4 = South, 5 = West | category |
| `glazing_area_ratio` | Window area as a fraction of floor area (0, 0.10, 0.25, 0.40) | – |
| `glazing_distribution` | How windows are spread over the façades (0 = none, 1 = uniform, 2–5 = mostly one side) | category |
| `heating_load_kwh_m2` | Yearly heating load | kWh/m² |
| `cooling_load_kwh_m2` | Yearly cooling load | kWh/m² |

**C1 (4 pts).** Load the file into a DataFrame called `bldg`, print its shape, and show its first 5 rows.

Remember: Colab automatically shows only the **last line** of a cell. Anything before it must be shown with
`print(...)`. That is why `bldg.head()` stays on the last line below.

> 💡 **Hint:** Same as Lab 02: `pd.read_csv(get_data("building_energy_efficiency.csv"))`, then `.shape` and `.head()`.

In [ ]:
# TODO (C1): replace None
bldg = None
print(bldg.shape)
bldg.head()   # keep this last: only the last line of a cell is shown automatically

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert bldg is not None, "bldg is still None. Finish C1 first: bldg = pd.read_csv(get_data(...))."
assert bldg.shape == (768, 10), f"bldg should be 768 rows × 10 columns, but it is {bldg.shape}. Check the file name."
print("Correct!")

**C2 (5 pts).** Show summary statistics with `.describe()`.

> 💡 **Hint:** One line: `bldg.describe().round(2)`.

In [ ]:
# TODO (C2)

**✍️ Question:** (a) What are the mean, minimum, and maximum heating load (with units)? (b) Two columns are **categories stored as numbers**. Which ones, and why is the *mean* of such a column meaningless? *(5 pts)*

*Your answer:* 

**C3 (6 pts).** How many buildings are 7 m tall? Then use `groupby` to find the mean heating load for each height.

> 💡 **Hint:** Filter with `bldg[bldg["overall_height_m"] == 7]` and use `len(...)`. For the means: `bldg.groupby("overall_height_m")["heating_load_kwh_m2"].mean()`.

In [ ]:
# TODO (C3)
n_tall = None              # number of buildings with overall_height_m == 7
heat_by_height = None      # mean heating load for each height (use groupby)
print("7 m tall buildings:", n_tall)
heat_by_height

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert n_tall is not None and heat_by_height is not None, "n_tall or heat_by_height is still None. Finish C3 first."
assert n_tall == 384, f"Expected 384 buildings that are 7 m tall, but n_tall = {n_tall}. Filter on overall_height_m == 7 and count the rows with len(...)."
assert abs(heat_by_height[7.0] - 31.28) < 0.01 and abs(heat_by_height[3.5] - 13.34) < 0.01,     "Expected mean heating loads of about 13.34 (3.5 m) and 31.28 kWh/m² (7 m). Group by 'overall_height_m' and take the 'heating_load_kwh_m2' column."
print("Correct!")

**C4 (6 pts).** Use `groupby` on `glazing_area_ratio` to get the mean **heating** and **cooling** load for each glazing level, then describe the trend.

> 💡 **Hint:** Select two columns after the groupby with a list: `bldg.groupby("glazing_area_ratio")[["heating_load_kwh_m2", "cooling_load_kwh_m2"]].mean()`.

In [ ]:
# TODO (C4)
loads_by_glazing = None
loads_by_glazing

**✍️ Question:** How do heating and cooling loads change as the glazing ratio goes from 0 to 0.40? Give one physical reason why more window area could raise **both**. *(3 pts)*

*Your answer:* 

**C5 (6 pts).** Plot a histogram of the heating load (20 bins) with a title and axis labels with units.

> 💡 **Hint:** Copy the histogram cell from Lab 02 and change the column and labels.

In [ ]:
# TODO (C5): replace None and each "..."
plt.figure(figsize=(7, 4))
plt.hist(None, bins=20, edgecolor="white")   # the heating-load column
plt.xlabel("...")                            # include units
plt.ylabel("...")
plt.title("...")
plt.show()

**✍️ Question:** The histogram has **two humps**. Using your answer to C3, explain why. *(3 pts)*

*Your answer:* 

**C6 (8 pts).** Make a scatter plot of heating load (y) vs relative compactness (x), with the two heights
in **different colors** and a legend. Tip: filter the table into two parts and call `plt.scatter` twice, each with a
`label=`.

> 💡 **Hint:** `short = bldg[bldg["overall_height_m"] == 3.5]` and the same for 7. Then `plt.scatter(short["relative_compactness"], short["heating_load_kwh_m2"], label="3.5 m tall")` and again for `tall`. Finish with `plt.legend()`.

In [ ]:
# TODO (C6): replace each None and write the two plt.scatter lines
short = None   # only the buildings that are 3.5 m tall
tall = None    # only the buildings that are 7 m tall
plt.figure(figsize=(7, 4.5))
# plt.scatter(...)   <- short buildings, with label="3.5 m tall"
# plt.scatter(...)   <- tall buildings, with label="7 m tall" (marker="^" helps if you print in black and white)
plt.xlabel("Relative compactness (–)")
plt.ylabel("Heating load (kWh/m²)")
plt.title("Heating load vs relative compactness, by building height")
plt.legend()
plt.show()

**✍️ Question:** Describe the pattern. Overall, more compact buildings in this dataset have a *higher* heating load. In real buildings, a more compact shape (less envelope area for the same volume) usually *loses less* heat. How can both be true? What does this warn you about when reading a scatter plot? *Hint: compare the trend **within** one color with the trend **across all** the dots together.* *(4 pts)*

*Your answer:* 

**C7 (5 pts).** You want to predict **heating load** from design choices. Make `X` (all 8 design columns)
and `y` (heating load), and print their shapes.

> 💡 **Hint:** Make a list of the 8 column names, then `X = bldg[that_list]` and `y = bldg["heating_load_kwh_m2"]`.

In [ ]:
# TODO (C7)
X = None
y = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert X is not None and y is not None, "X or y is still None. Finish C7 first."
assert "cooling_load_kwh_m2" not in X.columns and "heating_load_kwh_m2" not in X.columns, "X must not contain either load column — they are outputs, not design choices."
assert X.shape == (768, 8), f"X should be 768 rows × 8 columns (the design columns), but it is {X.shape}."
assert y.shape == (768,), f"y should be the single heating-load column, shape (768,), but it is {y.shape}. Use single brackets."
print("Correct!  X:", X.shape, "  y:", y.shape)

**✍️ Question:** Why should `cooling_load_kwh_m2` **not** be one of the features? *(2 pts)*

*Your answer:* 

## Part D — Reflection: AI in your own studies (18 points · AI-Free)
Write **120–200 words** in your own voice. Address all three:
1. One task in your CE/ARCE courses, internship, or future job where an AI method could help. Which kind of learning
   (Part B) would it use, and what would the inputs and output be?
2. One place where you would **not** trust an AI output without checking — and how you would check it.
3. What data would the method need, and one data-quality problem you would watch for (think of Lab 02 and Part C).

*Your reflection:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.